# 3 Topic Modeling and Clustering

In this section, we will do following steps to do clustering and give them labels:
1. **Embedding**: Get embeddings of the summary using SentenceTransformer;
2. **Clustering**: Perform dimension reduction clustering using HDBScan or KMeans;
3. **Labeling**: Label the clusters;

In [1]:
import json
import numpy as np
import os
import pandas as pd
import polars as pl
import torch
import zipfile

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from embedding import get_embedder
from gemini_utility import GeminiInstructJsonClient
from hdbscan import HDBSCAN
from pydantic import BaseModel
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score

# if cuda is available, use UMAP in cuml.manifold
if torch.cuda.is_available():
    from cuml.manifold import UMAP
    CUDA_UMAP = True
else:
    from umap import UMAP
    CUDA_UMAP = False

## 3.1 Embedding Preparation

In [2]:
# summary_df = pl.read_parquet("021_filtered_data.parquet").sample(1000, seed=42)
summary_df = pl.read_parquet("021_filtered_data.parquet")
texts = summary_df.get_column('summary').to_list()
print("Number of records:", len(texts))

Number of records: 163071


In [3]:
embedding_model = get_embedder(test_mode=True)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [4]:
# Uncoment below to compute and save embeddings, which can be time-consuming. 
# We will load pre-computed embeddings in the next cell.

# embeddings = embedding_model.encode(
#     texts,
#     batch_size=64,
#     show_progress_bar=True
# )

# # save cache
# np.save('022_summary_embeddings.npy', embeddings)

In [5]:
# load cache
embeddings = np.load('022_summary_embeddings.npy')
embeddings.shape

(163071, 1024)

## 3.2 Grid Search for Clustering

In [6]:
# The following code was run on 10000 samples to find a good set of parameters
# Uncomment the cell to reproduce

# TARGET_OUTLIER = 0.2

# def eval_run(topics, emb):
#     labels = np.array(topics)
#     outlier_rate = np.mean(labels == -1)
#     valid = labels != -1
#     n_topics = len(set(labels[valid])) if valid.any() else 0

#     sil = np.nan
#     if valid.sum() > 20 and n_topics > 1:
#         sil = silhouette_score(emb[valid], labels[valid])

#     outlier_penalty = abs(outlier_rate - TARGET_OUTLIER)
#     combined_score = (sil * 0.7) - (outlier_penalty * 0.3) if not np.isnan(sil) else -outlier_penalty

#     return {
#         "n_topics": n_topics,
#         "outlier_rate": float(outlier_rate),
#         "silhouette": float(sil) if not np.isnan(sil) else np.nan,
#         "combined_score": float(combined_score) if not np.isnan(combined_score) else np.nan,
#     }

# param_grid = {
#     "n_neighbors": [40, 60],
#     "min_dist": [0.0],
#     "n_components": [5],
#     "min_cluster_size": [10, 20],
#     "min_samples": [1, 2],
#     "cluster_selection_epsilon": [0.0, 0.03],
# }


# results = []
# vectorizer_model = CountVectorizer(
#     stop_words="english",
#     ngram_range=(1, 2),
#     min_df=5,
#     max_df=0.90,
#     token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
# )

# all_settings = list(product(
#     param_grid["n_neighbors"],
#     param_grid["min_dist"],
#     param_grid["n_components"],
#     param_grid["min_cluster_size"],
#     param_grid["min_samples"],
#     param_grid["cluster_selection_epsilon"],
# ))

# total_runs = len(all_settings)
# for run_id, (n_neighbors, min_dist, n_components, min_cluster_size, min_samples, cluster_selection_epsilon) in enumerate(all_settings, start=1):
#     print(f"\n[{run_id}/{total_runs}] start -> nn={n_neighbors}, md={min_dist}, nc={n_components}, mcs={min_cluster_size}, ms={min_samples}, cse={cluster_selection_epsilon}")

#     umap_model = UMAP(
#         n_neighbors=n_neighbors,
#         n_components=n_components,
#         min_dist=min_dist,
#         metric="cosine",
#         random_state=42,
#         low_memory=True,
#     )
#     hdbscan_model = HDBSCAN(
#         min_cluster_size=min_cluster_size,
#         min_samples=min_samples,
#         metric="euclidean",
#         cluster_selection_method="eom",
#         cluster_selection_epsilon=cluster_selection_epsilon,
#         prediction_data=False,
#     )

#     tm = BERTopic(
#         umap_model=umap_model,
#         hdbscan_model=hdbscan_model,
#         vectorizer_model=vectorizer_model,
#         calculate_probabilities=False,
#         verbose=False,
#     )

#     topics, _ = tm.fit_transform(texts, embeddings)
#     m = eval_run(topics, embeddings)
#     m.update({
#         "n_neighbors": n_neighbors,
#         "min_dist": min_dist,
#         "n_components": n_components,
#         "min_cluster_size": min_cluster_size,
#         "min_samples": min_samples,
#         "cluster_selection_epsilon": cluster_selection_epsilon,
#     })
#     results.append(m)
#     print(f"[{run_id}/{total_runs}] done  -> topics={m['n_topics']}, outlier={m['outlier_rate']:.4f}, sil={m['silhouette']:.4f}")

# res_df = pd.DataFrame(results)
# res_df["topic_range_score"] = res_df["n_topics"].apply(
#     lambda x: 1 if 30 <= x <= 90 else (0.5 if 20 <= x < 30 or 90 < x <= 120 else 0)
# )

# res_df = res_df.sort_values(
#     by=["combined_score", "topic_range_score", "silhouette"],
#     ascending=[False, False, False],
# ).reset_index(drop=True)

## 3.3 Clustering on Full Dataset

After tuning, we choose the following parameters for clustering.

UMAP:
- `n_neighbors=40`;
- `n_components=5`;
- `min_dist=0.0`;

HDBScan:
- `min_cluster_size=20`;
- `min_samples=1`;

In [7]:
umap_model = UMAP(
    n_neighbors=40,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    min_samples=1,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=False
)

vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
)
ctfidf_model = ClassTfidfTransformer(bm25_weighting=True, reduce_frequent_words=True)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=False
)

In [8]:
topics, probs = topic_model.fit_transform(texts, embeddings)
topic_info = topic_model.get_topic_info()

print(f"Number of topics: {topic_info.shape[0] - 1}")
topic_info.head(10)

Number of topics: 1712


,Topic,Count,Name,Representation,Representative_Docs
0,-1,56704,-1_models llms_domain_bad_meme,"[models llms, domain, bad, meme, hospital, ai ...",[Twister and Pictionary have launched new AI-p...
1,0,1028,0_million seed_funding led_seed funding_millio...,"[million seed, funding led, seed funding, mill...","[Vodex, a Generative AI company specializing i..."
2,1,1016,1_event feature_event aims_networking opportun...,"[event feature, event aims, networking opportu...",[DataGlobal Hub is hosting the Global Data & A...
3,2,1000,2_smiling_adult woman_adult_portrait,"[smiling, adult woman, adult, portrait, featur...",[This content is an advertisement for an AI-ge...
4,3,939,3_academic integrity_assignments_cheating_educ...,"[academic integrity, assignments, cheating, ed...","[ChatGPT, an AI chatbot, is raising concerns a..."
5,4,626,4_nvidia stock_nvidia reported_driven high_chi...,"[nvidia stock, nvidia reported, driven high, c...",[Nvidia reported a significant surge in revenu...
6,5,576,5_darktrace_vectra_vectra ai_detection response,"[darktrace, vectra, vectra ai, detection respo...",[SentinelOne has expanded its AI Security Plat...
7,6,562,6_breast_breast cancer_lung_cancer detection,"[breast, breast cancer, lung, cancer detection...",[Researchers are using artificial intelligence...
8,7,518,7_edge ai_jetson_nvidia jetson_low power,"[edge ai, jetson, nvidia jetson, low power, br...",[Leopard Imaging is launching its new LI-AGO-B...
9,8,506,8_heygen_ai avatars_voiceovers_scripts,"[heygen, ai avatars, voiceovers, scripts, vide...",[HeyGen offers an AI-powered platform to effor...


In [9]:
fig = topic_model.visualize_barchart(top_n_topics=10)
fig.show()

In [10]:
summary_df = summary_df.with_columns(pl.Series("cluster", topics))
summary_df.head(10)

date,title,summary,organization,industry,impact,technology,cluster
date,str,str,str,str,i8,str,i64
2025-06-23,"""Bad Idea AI Price (BAD), Marke…","""Bad Idea AI (BAD) is a cryptoc…","""Bad Idea AI""","""Cryptocurrency""",0,"""AI""",-1
2024-07-01,"""This AI video of gymnastics mi…","""AI-generated videos of gymnast…","""Luma Dream Machine""","""Media and Entertainment""",-2,"""AI-generated video""",-1
2024-09-22,"""If using AI feels like a chore…","""1minAI offers a lifetime subsc…","""1minAI""","""Software""",1,"""GPT-4, Gemini Pro, Claude""",1292
2023-11-10,"""The Road Ahead: How China's AI…","""China's AI Foundation Model, d…","""Baidu""","""Automotive""",2,"""deep learning, big data, cloud…",-1
2023-11-19,"""Microsoft and Nvidia to Empowe…","""Microsoft and Nvidia are colla…","""Microsoft and Nvidia""","""Technology""",1,"""Windows AI Studio and TensorRT…",-1
2023-12-12,"""Google Releases New Chatbot Ba…","""Google has launched its new AI…","""Google""","""Technology""",1,"""Gemini large language model""",-1
2023-09-07,"""Zoom Expands AI Offering with …","""Zoom is enhancing its platform…","""Zoom""","""Technology""",1,"""Generative AI""",187
2023-08-04,"""Pro-AI Thinking: Enhancing Ind…","""Pro-AI thinking integrates hum…","""Industry 5.0""","""Industrial Operations""",2,"""AI""",-1
2024-03-13,"""Best AI Prompts for Business R…","""ClickUp AI offers powerful too…","""ClickUp""","""Business Software""",1,"""AI""",-1


In [ ]:
# Checkpoint save
summary_df.write_parquet("023_clustered_data.parquet")

## 3.4 Label Clusters using GenAI

Although traditional techniques like TF-IDF can be used to extract keywords for each cluster, we will leverage GenAI to generate more human-friendly labels for each cluster. We will provide some sample summaries from each cluster to the GenAI model and ask it to generate a concise topic label that captures the essence of those summaries.

In [ ]:
# # Checkpoint load
# summary_df = pl.read_parquet("023_clustered_data.parquet")

cluster_samples = []

for idx, sub_df in summary_df.to_pandas().groupby('cluster'):
    if idx == -1:
        continue
    sample_texts = sub_df['summary'].sample(5).tolist()
    cluster_samples.append((idx, '\n'.join(sample_texts)))

len(cluster_samples)

1712

In [15]:
resp_cache_dir = "topics"

In [ ]:
# Uncomment the following code in this cell to run the labeling

# class TopicSummary(BaseModel):
#     topic: str
    

# instruction = '''
# Answer with a brief, short phrase summarizing given samples of some article summaries:
# '''.strip() + '\n\n'


# instruct_client = GeminiInstructJsonClient(
#     llm_model='gemini-2.5-flash-lite',
#     instruction=instruction,
#     resp_model=TopicSummary
# )

# instruct_client.execute_and_save_cache(
#     cache_dir=resp_cache_dir,
#     texts=[x[1] for x in cluster_samples],
#     batch_size=200,
#     max_workers=48
# )

# # archive the cache files into a zip

# with zipfile.ZipFile('topics.zip', 'w') as zipf:
#     for root, _, files in os.walk(resp_cache_dir):
#         for file in files:
#             zipf.write(os.path.join(root, file), arcname=file)

In [16]:
# Load from cache

with zipfile.ZipFile('topics.zip', 'r') as zipf:
    zipf.extractall(resp_cache_dir)

cache_files = sorted(list(os.listdir(resp_cache_dir)))
summary_responses = []
error_count = 0

for file in cache_files:
    with open(os.path.join(resp_cache_dir, file), 'r') as f:
        for line in f:
            res = json.loads(line)
            if res[0]:
                error_count += 1
            summary_responses.append(res)

print(f"Total summaries: {len(summary_responses)}, Errors: {error_count}")

Total summaries: 1712, Errors: 0


In [17]:
topic_map = {-1: ''}
for (idx, _), res in zip(cluster_samples, summary_responses):
    topic_map[idx] = res[1]['topic']

summary_df = summary_df.with_columns(
    pl.col('cluster').replace_strict(topic_map, return_dtype=pl.Utf8).alias('topic')
)
summary_df.head(5)

date,title,summary,organization,industry,impact,technology,cluster,topic
date,str,str,str,str,i8,str,i64,str
2025-06-23,"""Bad Idea AI Price (BAD), Marke…","""Bad Idea AI (BAD) is a cryptoc…","""Bad Idea AI""","""Cryptocurrency""",0,"""AI""",-1,""""""
2024-07-01,"""This AI video of gymnastics mi…","""AI-generated videos of gymnast…","""Luma Dream Machine""","""Media and Entertainment""",-2,"""AI-generated video""",-1,""""""
2024-09-22,"""If using AI feels like a chore…","""1minAI offers a lifetime subsc…","""1minAI""","""Software""",1,"""GPT-4, Gemini Pro, Claude""",1292,"""AI tools and services with lif…"
2023-11-10,"""The Road Ahead: How China's AI…","""China's AI Foundation Model, d…","""Baidu""","""Automotive""",2,"""deep learning, big data, cloud…",-1,""""""
2023-11-19,"""Microsoft and Nvidia to Empowe…","""Microsoft and Nvidia are colla…","""Microsoft and Nvidia""","""Technology""",1,"""Windows AI Studio and TensorRT…",-1,""""""


In [18]:
# Save checkpoint
summary_df.write_parquet("024_labeled_cluster_data.parquet")